# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.


In [ ]:
# List available record sets and their descriptions using their @ids
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"  @id: {rs['@id']} | name: {rs.get('name', 'N/A')} | description: {rs.get('description', '')}")

# Display fields (columns) for all record sets
for rs in metadata.record_sets:
    print(f"\nFields in RecordSet @{rs['@id']}")
    for field in rs['fields']:
        field_id = field['@id']
        field_name = field.get('name', '')
        field_dtype = field.get('dataType', '')
        print(f"  Field @id: {field_id} | name: {field_name} | dataType: {field_dtype}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.


In [ ]:
# List all record set IDs. Select key record set to explore.
record_sets_ids = [rs['@id'] for rs in metadata.record_sets]
# For this dataset, there's one main data record set:
main_record_set_id = record_sets_ids[0] if record_sets_ids else None

print(f"Record sets found: {record_sets_ids}")
dataframes = {}

for record_set in record_sets_ids:
    print(f"\\nLoading records for record set: {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} records. Sample columns: {df.columns[:5].tolist()}")

if main_record_set_id is not None:
    print("\nColumns in main record set:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.


In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Identify a suitable numeric field for analysis by searching for common columns
numeric_candidates = ['cr:TotalPeptides', 'cr:MW', 'cr:pI', 'cr:abundance', 'cr:Coverage']
df = dataframes[main_record_set_id]
numeric_field = next((col for col in numeric_candidates if col in df.columns), None)
print("Using numeric field for analysis:", numeric_field)

if numeric_field is not None:
    # Filter records where numeric_field > its mean
    try:
        field_series = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = field_series.mean()
        filtered_df = df[field_series > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df[[numeric_field]].head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (field_series[field_series > threshold] - field_series.mean()) / field_series.std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field—typical candidates: 'cr:Description', 'cr:Accession'
        group_candidates = ['cr:Description', 'cr:Sample']
        group_field = next((col for col in group_candidates if col in df.columns), None)
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    except Exception as e:
        print(f"Could not perform numeric operations: {e}")
else:
    print("No numeric field in data for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt

if numeric_field is not None and not df[numeric_field].isnull().all():
    plt.figure(figsize=(8, 5))
    field_series = pd.to_numeric(df[numeric_field], errors='coerce').dropna()
    plt.hist(field_series, bins=30, alpha=0.7, color='steelblue')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()
    
    # Visualize mean per group if grouped_df exists
    if 'grouped_df' in locals() and group_field is not None:
        plt.figure(figsize=(10, 5))
        grouped_df.plot(kind='bar', x=group_field, y=numeric_field, legend=False, color='teal', ax=plt.gca())
        plt.ylabel(f'Mean {numeric_field}')
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.tight_layout()
        plt.show()
else:
    print('No suitable numeric field found for visualization.')

## 6. Conclusion
In this notebook, we explored the FAIR<sup>2</sup> protein abundance dataset published under DOI [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa/).

Key findings:
- We programmatically loaded the dataset via its Croissant schema, reviewed its data structure, and extracted its main record set using their `@id` values.
- Fields include protein attributes such as total peptides, molecular weight, pI, and abundance. Filtering, normalization, and grouping operations were demonstrated.
- Distributions and group means were visualized to highlight variation across main protein or sample attributes.

Further analysis can involve inter-field correlations or integrating annotation metadata, as all record set and field `@id`s are available for reference.
